# GlAD-SCN — Architecture Benchmark Notebook
### Graph-Level Anomaly Detection

---
Structured Jupyter Notebook converted from your script.


In [ ]:
"""
GLAD-SCN: Graph-Level Anomaly Detection via Stable ChebNet
===========================================================
Architecture: Dual-encoder + Graph Autoencoder + Contrastive Loss
             as in Figure 3 of the GSoC proposal.

Models implemented (each fully self-contained):
  1. ChebNetGLAD        -- Vanilla ChebNet (K=3)
  2. StableChebNetGLAD  -- Stable-ChebNet (antisymmetric W + forward Euler)
  3. GCNGLAD            -- GCN backbone
  4. EdgeConvGLAD       -- 2-layer EdgeConv encoder
  5. GraphSAGEGLAD      -- 3-layer GraphSAGE encoder
  6. GATRewiredGLAD     -- GAT with long-range rewiring (DRew-style virtual edges)

Training : SM jets only  (unsupervised)
Evaluation: SM + BSM mixed  (labels used only for AUC / metrics)
"""

import os, math, time, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import (
    ChebConv, GCNConv, SAGEConv, GATConv,
    global_mean_pool, global_max_pool, global_add_pool,
    MessagePassing,
)
from torch_geometric.nn import knn_graph
from torch_geometric.utils import add_self_loops, to_dense_adj
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------
# 0. GLOBAL CONFIG
# -----------------------------------------------------------------
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
K_CHEB      = 3           # Chebyshev polynomial order
HIDDEN      = 64          # hidden dimension
PROJ_DIM    = 32          # projection-head output dimension
EPOCHS      = 80
LR          = 3e-4
BATCH_SIZE  = 32
TAU         = 0.5         # SimCLR temperature
ETA         = 0.1         # weight perturbation coefficient
SIGMA       = 0.1         # std of weight-noise
LAMBDA1     = 0.5         # contrastive loss weight
LAMBDA2     = 0.5         # representation-error loss weight
EPSILON     = 0.5         # Euler step size (Stable-ChebNet)
GAMMA       = 0.05        # dissipation force   (Stable-ChebNet)
TEST_FRAC   = 0.20        # fraction of SM used for test (mixed with BSM)
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")


# -----------------------------------------------------------------
# 1. DATA LOADING & SPLITTING
# -----------------------------------------------------------------

def load_dataset(path=r"dataset\jets_processed.pt"):
    """
    Load preprocessed graphs.
    File: dict with keys 'pyg_sm' (label 0) and 'pyg_bsm' (label 1).
    Returns: train_loader (SM only), test_loader (SM + BSM mixed).
    """
    raw = torch.load(path, weights_only=False)
    sm_graphs  = raw["pyg_sm"]
    bsm_graphs = raw["pyg_bsm"]

    for g in sm_graphs:
        g.y = torch.tensor([0], dtype=torch.long)
    for g in bsm_graphs:
        g.y = torch.tensor([1], dtype=torch.long)

    idx_sm    = torch.randperm(len(sm_graphs))
    sm_graphs = [sm_graphs[i] for i in idx_sm]

    n_test_sm = int(len(sm_graphs) * TEST_FRAC)
    train_sm  = sm_graphs[n_test_sm:]
    test_sm   = sm_graphs[:n_test_sm]
    test_data = test_sm + list(bsm_graphs)

    print(f"  Train (SM only): {len(train_sm)} jets")
    print(f"  Test  (SM+BSM) : {len(test_data)} jets  "
          f"({len(test_sm)} SM + {len(bsm_graphs)} BSM)")

    train_loader = DataLoader(train_sm,  batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
    test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    return train_loader, test_loader


# -----------------------------------------------------------------
# 2. SHARED BUILDING BLOCKS
# -----------------------------------------------------------------

def global_pool_concat(x, batch):
    """Concat mean, max, min, sum pooling -> (B, 4*hidden)."""
    mean_p = global_mean_pool(x, batch)
    max_p  = global_max_pool(x, batch)
    sum_p  = global_add_pool(x, batch)
    min_p  = -global_max_pool(-x, batch)
    return torch.cat([mean_p, max_p, min_p, sum_p], dim=-1)


class MLPProjectionHead(nn.Module):
    """2-layer MLP: graph embedding -> contrastive projection space."""
    def __init__(self, in_dim, proj_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.BatchNorm1d(in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)


def nt_xent_loss(z1, z2, temperature=TAU):
    """
    SimCLR NT-Xent loss.
    z1, z2: (N, D) graph-level projections from clean & perturbed encoders.
    """
    N  = z1.size(0)
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    z  = torch.cat([z1, z2], dim=0)                     # (2N, D)
    sim = torch.mm(z, z.T) / temperature                 # (2N, 2N)

    pos_sim = torch.cat([torch.diag(sim, N), torch.diag(sim, -N)], dim=0)  # (2N,)
    mask    = ~torch.eye(2*N, dtype=torch.bool, device=z.device)
    neg_sim = sim[mask].view(2*N, -1)                   # (2N, 2N-1)

    loss = -pos_sim + torch.logsumexp(
        torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1), dim=1
    )
    return loss.mean()


def structure_decoder(z_node, edge_index):
    """Inner-product structure decoder -> sigmoid edge scores."""
    src, dst = edge_index
    logits   = (z_node[src] * z_node[dst]).sum(dim=-1)
    return torch.sigmoid(logits)


def dirichlet_energy(x, edge_index, batch):
    """
    E = sum_{(i,j) in E} ||x_i - x_j||^2 / |E|  (per-graph mean).
    Higher value => more feature diversity => less over-smoothing.
    """
    src, dst = edge_index
    diff     = x[src] - x[dst]
    energy   = (diff ** 2).sum(dim=-1)
    num_graphs = int(batch.max().item()) + 1
    de     = torch.zeros(num_graphs, device=x.device)
    counts = torch.zeros(num_graphs, device=x.device)
    de.scatter_add_(0, batch[src], energy)
    counts.scatter_add_(0, batch[src], torch.ones_like(energy))
    return (de / counts.clamp(min=1)).mean().item()


def perturb_weights(module, eta=ETA, sigma=SIGMA):
    """Add Gaussian noise eta*N(0,sigma^2) to all parameters in-place."""
    with torch.no_grad():
        for p in module.parameters():
            p.data.add_(eta * torch.randn_like(p.data) * sigma)


# -----------------------------------------------------------------
# 3.A  ChebNet GLAD
# -----------------------------------------------------------------

class ChebEncoderBlock(nn.Module):
    """3 x ChebConv -> BN -> ReLU -> global pool concat."""
    def __init__(self, in_channels, hidden, K=K_CHEB):
        super().__init__()
        self.conv1 = ChebConv(in_channels, hidden, K)
        self.conv2 = ChebConv(hidden,      hidden, K)
        self.conv3 = ChebConv(hidden,      hidden, K)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.bn3   = nn.BatchNorm1d(hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        return x, global_pool_concat(x, batch)


class ChebAttrDecoder(nn.Module):
    """2 x ChebConv attribute decoder."""
    def __init__(self, hidden, out_channels, K=K_CHEB):
        super().__init__()
        self.conv1 = ChebConv(hidden, hidden,       K)
        self.conv2 = ChebConv(hidden, out_channels, K)

    def forward(self, z, edge_index):
        return self.conv2(F.relu(self.conv1(z, edge_index)), edge_index)


class ChebNetGLAD(nn.Module):
    """
    GLAD with Vanilla ChebNet backbone (K=3).
    Follows Figure 3 exactly:
      clean encoder f(theta)    -> ZG, z_node
      perturbed encoder f(theta')-> Z''G
      MLP head -> Z'_G, Z'_pert
      Graph Autoencoder (structure + attribute decoder) -> L_RC
      Re-encoder on G_hat -> L_RE
      Anomaly score = node RE + graph RE
    """
    def __init__(self, in_channels, hidden=HIDDEN, proj_dim=PROJ_DIM, K=K_CHEB):
        super().__init__()
        self.in_channels = in_channels
        self.hidden      = hidden
        self.K           = K
        self.encoder     = ChebEncoderBlock(in_channels, hidden, K)
        self.proj_head   = MLPProjectionHead(4 * hidden, proj_dim)
        self.attr_dec    = ChebAttrDecoder(hidden, in_channels, K)

    def encode(self, x, edge_index, batch):
        return self.encoder(x, edge_index, batch)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # --- Clean encoder ---
        z_node, Z_G = self.encode(x, edge_index, batch)
        Z_prime_G   = self.proj_head(Z_G)

        # --- Perturbed encoder (copy + noise) ---
        p_enc = ChebEncoderBlock(self.in_channels, self.hidden, self.K).to(x.device)
        p_enc.load_state_dict(self.encoder.state_dict())
        perturb_weights(p_enc)
        with torch.no_grad():
            _, Z_pp_G = p_enc(x, edge_index, batch)
        Z_prime_pert = self.proj_head(Z_pp_G)

        # --- Graph Autoencoder ---
        A_hat  = structure_decoder(z_node, edge_index)
        X_hat  = self.attr_dec(z_node, edge_index)
        L_RC   = (F.binary_cross_entropy(A_hat, torch.ones_like(A_hat))
                  + F.mse_loss(X_hat, x))

        # --- Contrastive loss ---
        L_CT = nt_xent_loss(Z_prime_G, Z_prime_pert)

        # --- Re-encoder on reconstructed graph ---
        z_node_r, Z_G_r = self.encode(X_hat.detach(), edge_index, batch)
        L_RE = F.mse_loss(Z_G, Z_G_r) + F.mse_loss(z_node, z_node_r)

        L_total = L_CT + LAMBDA1 * L_RC + LAMBDA2 * L_RE

        # Anomaly score per graph
        per_node_re   = ((z_node - z_node_r) ** 2).sum(dim=-1)
        score_node    = global_add_pool(per_node_re.unsqueeze(-1), batch).squeeze(-1)
        anomaly_score = score_node + ((Z_G - Z_G_r) ** 2).sum(dim=-1)

        return {
            "loss":          L_total,
            "L_CT":          L_CT.item(),
            "L_RC":          L_RC.item(),
            "L_RE":          L_RE.item(),
            "anomaly_score": anomaly_score.detach(),
            "z_node":        z_node,
            "edge_index":    edge_index,
            "batch":         batch,
        }


# -----------------------------------------------------------------
# 3.B  Stable-ChebNet GLAD
# -----------------------------------------------------------------

class StableChebLayer(nn.Module):
    """
    Stable-ChebNet layer (Eq. 9 from paper):
      X^{l+1} = X^l + eps * sum_k T_k(L~) X^l (W_k - W_k^T - gamma*I)
    Antisymmetric parameterization enforces purely imaginary Jacobian eigenvalues
    -> second-order stability, no exponential growth/decay (Theorem 4).
    """
    def __init__(self, channels, K=K_CHEB, epsilon=EPSILON, gamma=GAMMA):
        super().__init__()
        self.K       = K
        self.epsilon = epsilon
        self.gamma   = gamma
        # Raw learnable weights; antisymmetric form: W - W^T
        self.W_raw = nn.ParameterList([
            nn.Parameter(torch.randn(channels, channels) * 0.01)
            for _ in range(K + 1)
        ])
        # One ChebConv per polynomial order to apply T_k(L~) X
        self.cheb_convs = nn.ModuleList([
            ChebConv(channels, channels, k + 1) for k in range(K + 1)
        ])
        self.bn = nn.BatchNorm1d(channels)

    def forward(self, x, edge_index):
        agg = torch.zeros_like(x)
        for k in range(self.K + 1):
            Tk_X = self.cheb_convs[k](x, edge_index)    # T_k(L~) X
            W_k  = self.W_raw[k]
            # Antisymmetric weight + dissipation: W - W^T - gamma*I
            W_as = W_k - W_k.T - self.gamma * torch.eye(
                W_k.size(0), device=W_k.device
            )
            agg = agg + Tk_X @ W_as
        # Forward Euler update: X^{l+1} = X^l + eps * agg
        x_new = x + self.epsilon * agg
        return F.relu(self.bn(x_new))


class StableChebEncoderBlock(nn.Module):
    """3 x StableChebLayer -> global pool concat."""
    def __init__(self, in_channels, hidden, K=K_CHEB):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden)
        self.layer1     = StableChebLayer(hidden, K)
        self.layer2     = StableChebLayer(hidden, K)
        self.layer3     = StableChebLayer(hidden, K)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.input_proj(x))
        x = self.layer1(x, edge_index)
        x = self.layer2(x, edge_index)
        x = self.layer3(x, edge_index)
        return x, global_pool_concat(x, batch)


class StableChebAttrDecoder(nn.Module):
    """2 x StableChebLayer -> linear out."""
    def __init__(self, hidden, out_channels, K=K_CHEB):
        super().__init__()
        self.layer1   = StableChebLayer(hidden, K)
        self.layer2   = StableChebLayer(hidden, K)
        self.out_proj = nn.Linear(hidden, out_channels)

    def forward(self, z, edge_index):
        x = self.layer1(z, edge_index)
        x = self.layer2(x, edge_index)
        return self.out_proj(x)


class StableChebNetGLAD(nn.Module):
    """
    GLAD with Stable-ChebNet backbone.
    Same Figure-3 architecture; encoder layers use antisymmetric forward-Euler update
    guaranteeing ||J||_2 = 1 + O(eps^2) (Theorem 4 from Hariri et al.).
    """
    def __init__(self, in_channels, hidden=HIDDEN, proj_dim=PROJ_DIM, K=K_CHEB):
        super().__init__()
        self.in_channels = in_channels
        self.hidden      = hidden
        self.K           = K
        self.encoder     = StableChebEncoderBlock(in_channels, hidden, K)
        self.proj_head   = MLPProjectionHead(4 * hidden, proj_dim)
        self.attr_dec    = StableChebAttrDecoder(hidden, in_channels, K)

    def encode(self, x, edge_index, batch):
        return self.encoder(x, edge_index, batch)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        z_node, Z_G  = self.encode(x, edge_index, batch)
        Z_prime_G    = self.proj_head(Z_G)

        p_enc = StableChebEncoderBlock(self.in_channels, self.hidden, self.K).to(x.device)
        p_enc.load_state_dict(self.encoder.state_dict())
        perturb_weights(p_enc)
        with torch.no_grad():
            _, Z_pp_G = p_enc(x, edge_index, batch)
        Z_prime_pert = self.proj_head(Z_pp_G)

        A_hat  = structure_decoder(z_node, edge_index)
        X_hat  = self.attr_dec(z_node, edge_index)
        L_RC   = (F.binary_cross_entropy(A_hat, torch.ones_like(A_hat))
                  + F.mse_loss(X_hat, x))

        L_CT = nt_xent_loss(Z_prime_G, Z_prime_pert)

        z_node_r, Z_G_r = self.encode(X_hat.detach(), edge_index, batch)
        L_RE    = F.mse_loss(Z_G, Z_G_r) + F.mse_loss(z_node, z_node_r)
        L_total = L_CT + LAMBDA1 * L_RC + LAMBDA2 * L_RE

        per_node_re   = ((z_node - z_node_r) ** 2).sum(dim=-1)
        score_node    = global_add_pool(per_node_re.unsqueeze(-1), batch).squeeze(-1)
        anomaly_score = score_node + ((Z_G - Z_G_r) ** 2).sum(dim=-1)

        return {
            "loss":          L_total,
            "L_CT":          L_CT.item(),
            "L_RC":          L_RC.item(),
            "L_RE":          L_RE.item(),
            "anomaly_score": anomaly_score.detach(),
            "z_node":        z_node,
            "edge_index":    edge_index,
            "batch":         batch,
        }


# -----------------------------------------------------------------
# 3.C  GCN GLAD
# -----------------------------------------------------------------

class GCNEncoderBlock(nn.Module):
    """3 x GCNConv -> BN -> ReLU -> global pool concat."""
    def __init__(self, in_channels, hidden):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden,      hidden)
        self.conv3 = GCNConv(hidden,      hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.bn3   = nn.BatchNorm1d(hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        return x, global_pool_concat(x, batch)


class GCNAttrDecoder(nn.Module):
    def __init__(self, hidden, out_channels):
        super().__init__()
        self.conv1 = GCNConv(hidden, hidden)
        self.conv2 = GCNConv(hidden, out_channels)

    def forward(self, z, edge_index):
        return self.conv2(F.relu(self.conv1(z, edge_index)), edge_index)


class GCNGLAD(nn.Module):
    """GLAD with GCN backbone."""
    def __init__(self, in_channels, hidden=HIDDEN, proj_dim=PROJ_DIM):
        super().__init__()
        self.in_channels = in_channels
        self.hidden      = hidden
        self.encoder     = GCNEncoderBlock(in_channels, hidden)
        self.proj_head   = MLPProjectionHead(4 * hidden, proj_dim)
        self.attr_dec    = GCNAttrDecoder(hidden, in_channels)

    def encode(self, x, edge_index, batch):
        return self.encoder(x, edge_index, batch)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        z_node, Z_G  = self.encode(x, edge_index, batch)
        Z_prime_G    = self.proj_head(Z_G)

        p_enc = GCNEncoderBlock(self.in_channels, self.hidden).to(x.device)
        p_enc.load_state_dict(self.encoder.state_dict())
        perturb_weights(p_enc)
        with torch.no_grad():
            _, Z_pp_G = p_enc(x, edge_index, batch)
        Z_prime_pert = self.proj_head(Z_pp_G)

        A_hat  = structure_decoder(z_node, edge_index)
        X_hat  = self.attr_dec(z_node, edge_index)
        L_RC   = (F.binary_cross_entropy(A_hat, torch.ones_like(A_hat))
                  + F.mse_loss(X_hat, x))
        L_CT   = nt_xent_loss(Z_prime_G, Z_prime_pert)

        z_node_r, Z_G_r = self.encode(X_hat.detach(), edge_index, batch)
        L_RE    = F.mse_loss(Z_G, Z_G_r) + F.mse_loss(z_node, z_node_r)
        L_total = L_CT + LAMBDA1 * L_RC + LAMBDA2 * L_RE

        per_node_re   = ((z_node - z_node_r) ** 2).sum(dim=-1)
        score_node    = global_add_pool(per_node_re.unsqueeze(-1), batch).squeeze(-1)
        anomaly_score = score_node + ((Z_G - Z_G_r) ** 2).sum(dim=-1)

        return {"loss": L_total, "L_CT": L_CT.item(), "L_RC": L_RC.item(),
                "L_RE": L_RE.item(), "anomaly_score": anomaly_score.detach(),
                "z_node": z_node, "edge_index": edge_index, "batch": batch}


# -----------------------------------------------------------------
# 3.D  EdgeConv GLAD  (2-layer encoder only, as specified in proposal)
# -----------------------------------------------------------------

class EdgeConvLayer(MessagePassing):
    """EdgeConv: h_i = max_{j in N(i)} MLP([h_i || h_j - h_i])"""
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr="max")
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels),
        )

    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)

    def message(self, x_i, x_j):
        return self.mlp(torch.cat([x_i, x_j - x_i], dim=-1))


class EdgeConvEncoderBlock(nn.Module):
    """2 x EdgeConv -> BN -> ReLU -> global pool concat  (proposal: 2-layer)."""
    def __init__(self, in_channels, hidden):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden)
        self.conv1      = EdgeConvLayer(hidden, hidden)
        self.conv2      = EdgeConvLayer(hidden, hidden)
        self.bn1        = nn.BatchNorm1d(hidden)
        self.bn2        = nn.BatchNorm1d(hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.input_proj(x))
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        return x, global_pool_concat(x, batch)


class EdgeConvAttrDecoder(nn.Module):
    def __init__(self, hidden, out_channels):
        super().__init__()
        self.conv1 = EdgeConvLayer(hidden, hidden)
        self.conv2 = EdgeConvLayer(hidden, out_channels)

    def forward(self, z, edge_index):
        return self.conv2(F.relu(self.conv1(z, edge_index)), edge_index)


class EdgeConvGLAD(nn.Module):
    """GLAD with 2-layer EdgeConv backbone."""
    def __init__(self, in_channels, hidden=HIDDEN, proj_dim=PROJ_DIM):
        super().__init__()
        self.in_channels = in_channels
        self.hidden      = hidden
        self.encoder     = EdgeConvEncoderBlock(in_channels, hidden)
        self.proj_head   = MLPProjectionHead(4 * hidden, proj_dim)
        self.attr_dec    = EdgeConvAttrDecoder(hidden, in_channels)

    def encode(self, x, edge_index, batch):
        return self.encoder(x, edge_index, batch)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        z_node, Z_G  = self.encode(x, edge_index, batch)
        Z_prime_G    = self.proj_head(Z_G)

        p_enc = EdgeConvEncoderBlock(self.in_channels, self.hidden).to(x.device)
        p_enc.load_state_dict(self.encoder.state_dict())
        perturb_weights(p_enc)
        with torch.no_grad():
            _, Z_pp_G = p_enc(x, edge_index, batch)
        Z_prime_pert = self.proj_head(Z_pp_G)

        A_hat  = structure_decoder(z_node, edge_index)
        X_hat  = self.attr_dec(z_node, edge_index)
        L_RC   = (F.binary_cross_entropy(A_hat, torch.ones_like(A_hat))
                  + F.mse_loss(X_hat, x))
        L_CT   = nt_xent_loss(Z_prime_G, Z_prime_pert)

        z_node_r, Z_G_r = self.encode(X_hat.detach(), edge_index, batch)
        L_RE    = F.mse_loss(Z_G, Z_G_r) + F.mse_loss(z_node, z_node_r)
        L_total = L_CT + LAMBDA1 * L_RC + LAMBDA2 * L_RE

        per_node_re   = ((z_node - z_node_r) ** 2).sum(dim=-1)
        score_node    = global_add_pool(per_node_re.unsqueeze(-1), batch).squeeze(-1)
        anomaly_score = score_node + ((Z_G - Z_G_r) ** 2).sum(dim=-1)

        return {"loss": L_total, "L_CT": L_CT.item(), "L_RC": L_RC.item(),
                "L_RE": L_RE.item(), "anomaly_score": anomaly_score.detach(),
                "z_node": z_node, "edge_index": edge_index, "batch": batch}


# -----------------------------------------------------------------
# 3.E  GraphSAGE GLAD  (3-layer encoder)
# -----------------------------------------------------------------

class SAGEEncoderBlock(nn.Module):
    """3 x SAGEConv -> BN -> ReLU -> global pool concat."""
    def __init__(self, in_channels, hidden):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden,      hidden)
        self.conv3 = SAGEConv(hidden,      hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.bn3   = nn.BatchNorm1d(hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        return x, global_pool_concat(x, batch)


class SAGEAttrDecoder(nn.Module):
    def __init__(self, hidden, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(hidden, hidden)
        self.conv2 = SAGEConv(hidden, out_channels)

    def forward(self, z, edge_index):
        return self.conv2(F.relu(self.conv1(z, edge_index)), edge_index)


class GraphSAGEGLAD(nn.Module):
    """GLAD with 3-layer GraphSAGE backbone."""
    def __init__(self, in_channels, hidden=HIDDEN, proj_dim=PROJ_DIM):
        super().__init__()
        self.in_channels = in_channels
        self.hidden      = hidden
        self.encoder     = SAGEEncoderBlock(in_channels, hidden)
        self.proj_head   = MLPProjectionHead(4 * hidden, proj_dim)
        self.attr_dec    = SAGEAttrDecoder(hidden, in_channels)

    def encode(self, x, edge_index, batch):
        return self.encoder(x, edge_index, batch)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        z_node, Z_G  = self.encode(x, edge_index, batch)
        Z_prime_G    = self.proj_head(Z_G)

        p_enc = SAGEEncoderBlock(self.in_channels, self.hidden).to(x.device)
        p_enc.load_state_dict(self.encoder.state_dict())
        perturb_weights(p_enc)
        with torch.no_grad():
            _, Z_pp_G = p_enc(x, edge_index, batch)
        Z_prime_pert = self.proj_head(Z_pp_G)

        A_hat  = structure_decoder(z_node, edge_index)
        X_hat  = self.attr_dec(z_node, edge_index)
        L_RC   = (F.binary_cross_entropy(A_hat, torch.ones_like(A_hat))
                  + F.mse_loss(X_hat, x))
        L_CT   = nt_xent_loss(Z_prime_G, Z_prime_pert)

        z_node_r, Z_G_r = self.encode(X_hat.detach(), edge_index, batch)
        L_RE    = F.mse_loss(Z_G, Z_G_r) + F.mse_loss(z_node, z_node_r)
        L_total = L_CT + LAMBDA1 * L_RC + LAMBDA2 * L_RE

        per_node_re   = ((z_node - z_node_r) ** 2).sum(dim=-1)
        score_node    = global_add_pool(per_node_re.unsqueeze(-1), batch).squeeze(-1)
        anomaly_score = score_node + ((Z_G - Z_G_r) ** 2).sum(dim=-1)

        return {"loss": L_total, "L_CT": L_CT.item(), "L_RC": L_RC.item(),
                "L_RE": L_RE.item(), "anomaly_score": anomaly_score.detach(),
                "z_node": z_node, "edge_index": edge_index, "batch": batch}


# -----------------------------------------------------------------
# 3.F  GAT Rewired GLAD
# -----------------------------------------------------------------
# Rewiring: add long-range k-NN edges in (eta, phi) space, DRew-style.
# This directly addresses over-squashing by connecting distant nodes
# in the jet substructure coordinate space.

class GATRewiredEncoderBlock(nn.Module):
    """
    3 x GATConv on rewired graph -> BN -> ELU -> global pool concat.
    Before each forward pass, long-range k-NN edges in (eta,phi) space
    are added to the original edge_index (DRew-style rewiring).
    """
    def __init__(self, in_channels, hidden, heads=4, k_long=4):
        super().__init__()
        self.k_long = k_long
        head_dim    = hidden // heads
        self.conv1  = GATConv(in_channels, head_dim, heads=heads, concat=True)
        self.conv2  = GATConv(hidden,      head_dim, heads=heads, concat=True)
        self.conv3  = GATConv(hidden,      head_dim, heads=heads, concat=True)
        self.bn1    = nn.BatchNorm1d(hidden)
        self.bn2    = nn.BatchNorm1d(hidden)
        self.bn3    = nn.BatchNorm1d(hidden)

    def _rewire(self, x, edge_index, batch):
        """Add long-range (eta,phi) k-NN edges. x: (N, in_ch); col 1,2 = eta,phi."""
        try:
            pos = x[:, 1:3]
            new_ei  = knn_graph(pos, k=self.k_long, batch=batch, loop=False)
            combined = torch.cat([edge_index, new_ei], dim=1)
            combined = torch.unique(combined, dim=1)
            return combined
        except Exception:
            return edge_index

    def forward(self, x, edge_index, batch):
        edge_index = self._rewire(x, edge_index, batch)
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        x = F.elu(self.bn3(self.conv3(x, edge_index)))
        return x, global_pool_concat(x, batch), edge_index


class GATRewiredAttrDecoder(nn.Module):
    def __init__(self, hidden, out_channels, heads=4):
        super().__init__()
        head_dim   = hidden // heads
        self.conv1 = GATConv(hidden, head_dim,   heads=heads, concat=True)
        self.conv2 = GATConv(hidden, out_channels, heads=1,  concat=False)

    def forward(self, z, edge_index):
        return self.conv2(F.elu(self.conv1(z, edge_index)), edge_index)


class GATRewiredGLAD(nn.Module):
    """
    GLAD with rewired GAT backbone.
    Rewiring adds long-range k-NN edges in (eta,phi) space before message passing,
    directly mitigating over-squashing in jet substructure tasks.
    """
    def __init__(self, in_channels, hidden=HIDDEN, proj_dim=PROJ_DIM,
                 heads=4, k_long=4):
        super().__init__()
        self.in_channels = in_channels
        self.hidden      = hidden
        self.heads       = heads
        self.k_long      = k_long
        self.encoder     = GATRewiredEncoderBlock(in_channels, hidden, heads, k_long)
        self.proj_head   = MLPProjectionHead(4 * hidden, proj_dim)
        self.attr_dec    = GATRewiredAttrDecoder(hidden, in_channels, heads)

    def encode(self, x, edge_index, batch):
        z_node, Z_G, ei_rw = self.encoder(x, edge_index, batch)
        return z_node, Z_G, ei_rw

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        z_node, Z_G, ei_rw = self.encode(x, edge_index, batch)
        Z_prime_G = self.proj_head(Z_G)

        p_enc = GATRewiredEncoderBlock(
            self.in_channels, self.hidden, self.heads, self.k_long
        ).to(x.device)
        p_enc.load_state_dict(self.encoder.state_dict())
        perturb_weights(p_enc)
        with torch.no_grad():
            _, Z_pp_G, _ = p_enc(x, edge_index, batch)
        Z_prime_pert = self.proj_head(Z_pp_G)

        # Decoder on original edge_index (pre-rewiring)
        A_hat  = structure_decoder(z_node, ei_rw)
        X_hat  = self.attr_dec(z_node, ei_rw)
        L_RC   = (F.binary_cross_entropy(A_hat, torch.ones_like(A_hat))
                  + F.mse_loss(X_hat, x))
        L_CT   = nt_xent_loss(Z_prime_G, Z_prime_pert)

        z_node_r, Z_G_r, _ = self.encode(X_hat.detach(), ei_rw, batch)
        L_RE    = F.mse_loss(Z_G, Z_G_r) + F.mse_loss(z_node, z_node_r)
        L_total = L_CT + LAMBDA1 * L_RC + LAMBDA2 * L_RE

        per_node_re   = ((z_node - z_node_r) ** 2).sum(dim=-1)
        score_node    = global_add_pool(per_node_re.unsqueeze(-1), batch).squeeze(-1)
        anomaly_score = score_node + ((Z_G - Z_G_r) ** 2).sum(dim=-1)

        return {
            "loss":          L_total,
            "L_CT":          L_CT.item(),
            "L_RC":          L_RC.item(),
            "L_RE":          L_RE.item(),
            "anomaly_score": anomaly_score.detach(),
            "z_node":        z_node,
            "edge_index":    ei_rw,
            "batch":         batch,
        }


# -----------------------------------------------------------------
# 4. TRAINING & EVALUATION
# -----------------------------------------------------------------

def train_epoch(model, loader, optimizer):
    model.train()
    total, L_CT_s, L_RC_s, L_RE_s, de_s = 0.0, 0.0, 0.0, 0.0, 0.0
    n = 0
    for data in loader:
        data = data.to(DEVICE)
        if data.x is None or data.x.size(0) == 0:
            continue
        optimizer.zero_grad()
        try:
            out = model(data)
        except Exception as e:
            print(f"  [skip batch] {e}")
            continue
        loss = out["loss"]
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total  += loss.item()
        L_CT_s += out["L_CT"]
        L_RC_s += out["L_RC"]
        L_RE_s += out["L_RE"]
        de_s   += dirichlet_energy(
            out["z_node"].detach(), out["edge_index"], out["batch"]
        )
        n += 1

    n = max(n, 1)
    return {"loss": total/n, "L_CT": L_CT_s/n, "L_RC": L_RC_s/n,
            "L_RE": L_RE_s/n, "dirichlet": de_s/n}


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_scores, all_labels = [], []
    for data in loader:
        data = data.to(DEVICE)
        if data.x is None or data.x.size(0) == 0:
            continue
        try:
            out = model(data)
        except Exception:
            continue
        scores = out["anomaly_score"].cpu().numpy()
        labels = data.y.cpu().numpy().flatten()
        ml     = min(len(scores), len(labels))
        all_scores.append(scores[:ml])
        all_labels.append(labels[:ml])

    if not all_scores:
        return {"auc_roc": 0.5, "auc_pr": 0.0, "scores": np.array([]), "labels": np.array([])}

    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels)
    return {
        "auc_roc": roc_auc_score(labels, scores),
        "auc_pr":  average_precision_score(labels, scores),
        "scores":  scores,
        "labels":  labels,
    }


# -----------------------------------------------------------------
# 5. TRAINING PIPELINE
# -----------------------------------------------------------------

def run_model(name, model, train_loader, test_loader):
    print(f"\n{'='*62}")
    print(f"  Training: {name}")
    print(f"{'='*62}")

    model     = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history   = {"loss": [], "L_CT": [], "L_RC": [], "L_RE": [], "dirichlet": []}
    best_auc  = 0.0
    t0        = time.time()

    for epoch in range(1, EPOCHS + 1):
        m = train_epoch(model, train_loader, optimizer)
        scheduler.step()
        for k, v in m.items():
            history[k].append(v)

        if epoch % 10 == 0 or epoch == 1:
            ev   = evaluate(model, test_loader)
            auc  = ev["auc_roc"]
            best_auc = max(best_auc, auc)
            print(
                f"  Ep {epoch:3d}/{EPOCHS} | "
                f"Loss={m['loss']:.4f}  "
                f"CT={m['L_CT']:.4f}  "
                f"RC={m['L_RC']:.4f}  "
                f"RE={m['L_RE']:.4f}  "
                f"DE={m['dirichlet']:.4f}  "
                f"AUC={auc:.4f}  "
                f"[{time.time()-t0:.0f}s]"
            )

    final_ev = evaluate(model, test_loader)
    print(f"\n  ★ {name}: AUC-ROC={final_ev['auc_roc']:.4f}  "
          f"AUC-PR={final_ev['auc_pr']:.4f}  BestAUC={best_auc:.4f}")

    return history, final_ev


# -----------------------------------------------------------------
# 6. PLOTTING
# -----------------------------------------------------------------

PALETTE = ["#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#F4A261", "#264653"]


def plot_results(all_histories, all_evals, model_names, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    # 6.1  Training Loss Curves
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Training Curves", fontsize=15, fontweight="bold")
    keys   = ["loss", "L_CT", "L_RC", "L_RE"]
    titles = ["Total Loss", "Contrastive Loss (L_CT)",
              "Reconstruction Loss (L_RC)", "Repr. Error (L_RE)"]
    for ax, key, title in zip(axes.flat, keys, titles):
        for name, hist, c in zip(model_names, all_histories, PALETTE):
            ax.plot(hist[key], label=name, color=c, linewidth=1.8)
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/training_curves.png", dpi=150)
    plt.close()

    # 6.2  Dirichlet Energy
    fig, ax = plt.subplots(figsize=(11, 5))
    for name, hist, c in zip(model_names, all_histories, PALETTE):
        ax.plot(hist["dirichlet"], label=name, color=c, linewidth=2.0)
    ax.set_title("Dirichlet Energy of Node Embeddings per Epoch\n"
                 "(Higher = more feature diversity = less over-smoothing)",
                 fontsize=12)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Dirichlet Energy")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/dirichlet_energy.png", dpi=150)
    plt.close()

    # 6.3  Anomaly Score Distributions
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Anomaly Score Distributions (SM vs BSM)", fontsize=14, fontweight="bold")
    for ax, name, ev in zip(axes.flat, model_names, all_evals):
        scores = ev.get("scores", np.array([]))
        labels = ev.get("labels", np.array([]))
        if len(scores) == 0:
            continue
        ax.hist(scores[labels == 0], bins=60, alpha=0.6, color="#457B9D",
                label="SM",  density=True)
        ax.hist(scores[labels == 1], bins=60, alpha=0.6, color="#E63946",
                label="BSM", density=True)
        ax.set_title(f"{name}\nAUC-ROC={ev.get('auc_roc', 0):.4f}", fontsize=10)
        ax.set_xlabel("Anomaly Score"); ax.set_ylabel("Density")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    for ax in axes.flat[len(model_names):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/anomaly_distributions.png", dpi=150)
    plt.close()

    # 6.4  AUC Bar Chart
    fig, ax = plt.subplots(figsize=(11, 5))
    aucs   = [ev.get("auc_roc", 0) for ev in all_evals]
    auc_pr = [ev.get("auc_pr",  0) for ev in all_evals]
    x = np.arange(len(model_names)); w = 0.35
    ax.bar(x - w/2, aucs,   w, label="AUC-ROC", color=PALETTE[:len(model_names)])
    ax.bar(x + w/2, auc_pr, w, label="AUC-PR",  color=PALETTE[:len(model_names)],
           alpha=0.55, hatch="//")
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=18, ha="right")
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
    ax.set_title("AUC-ROC and AUC-PR Comparison", fontsize=13)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    for i, v in enumerate(aucs):
        ax.text(i - w/2, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/auc_comparison.png", dpi=150)
    plt.close()

    # 6.5  Summary Table
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axis("off")
    de_finals = [h["dirichlet"][-1] if h["dirichlet"] else 0.0 for h in all_histories]
    table_data = [
        [name,
         f"{ev['auc_roc']:.4f}",
         f"{ev['auc_pr']:.4f}",
         f"{de:.4f}",
         "BETTER" if ev['auc_roc'] == max(aucs) else ""]
        for name, ev, de in zip(model_names, all_evals, de_finals)
    ]
    t = ax.table(
        cellText=table_data,
        colLabels=["Model", "AUC-ROC", "AUC-PR", "Dirichlet (final)", ""],
        cellLoc="center", loc="center",
    )
    t.auto_set_font_size(False)
    t.set_fontsize(9)
    t.scale(1.2, 1.9)
    # Highlight best row
    best_idx = aucs.index(max(aucs))
    for col in range(5):
        t[(best_idx + 1, col)].set_facecolor("#d4edda")
    ax.set_title("Results Summary", fontsize=13, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/results_table.png", dpi=150, bbox_inches="tight")
    plt.close()

    print(f"\n  All plots saved to ./{save_dir}/")


# -----------------------------------------------------------------
# 7. MAIN
# -----------------------------------------------------------------

def main():
    print("Loading dataset ...")
    train_loader, test_loader = load_dataset(r"dataset/jets_processed.pt")

    sample = next(iter(train_loader))
    in_ch  = sample.x.size(-1)
    print(f"  Input feature dim: {in_ch}")

    model_configs = [
        ("ChebNet-GLAD",       ChebNetGLAD(in_ch)),
        ("StableChebNet-GLAD", StableChebNetGLAD(in_ch)),
        ("GCN-GLAD",           GCNGLAD(in_ch)),
        ("EdgeConv-GLAD",      EdgeConvGLAD(in_ch)),
        ("GraphSAGE-GLAD",     GraphSAGEGLAD(in_ch)),
        ("GATRewired-GLAD",    GATRewiredGLAD(in_ch)),
    ]

    all_histories, all_evals, model_names = [], [], []

    for name, model in model_configs:
        hist, ev = run_model(name, model, train_loader, test_loader)
        all_histories.append(hist)
        all_evals.append(ev)
        model_names.append(name)

    # ----- Final Summary -----
    print("\n" + "="*62)
    print("  FINAL RESULTS SUMMARY")
    print("="*62)
    print(f"  {'Model':<25} {'AUC-ROC':>10} {'AUC-PR':>10} {'DE(final)':>12}")
    print("-"*62)
    for name, ev, hist in zip(model_names, all_evals, all_histories):
        de = hist["dirichlet"][-1] if hist["dirichlet"] else 0.0
        print(f"  {name:<25} {ev['auc_roc']:>10.4f} "
              f"{ev['auc_pr']:>10.4f} {de:>12.4f}")

    print("\n  DIRICHLET ENERGY ANALYSIS")
    print("  Higher DE = more feature diversity = less over-smoothing")
    print("-"*62)
    for name, hist in zip(model_names, all_histories):
        de = hist["dirichlet"]
        if de:
            print(f"  {name:<25}  start={de[0]:.4f}  end={de[-1]:.4f}  "
                  f"delta={de[-1]-de[0]:+.4f}  mean={np.mean(de):.4f}")

    plot_results(all_histories, all_evals, model_names)

    os.makedirs("outputs", exist_ok=True)
    for (name, model), ev in zip(model_configs, all_evals):
        safe_name = name.replace(" ", "_").replace("/", "-")
        torch.save({"model_state": model.state_dict(),
                    "auc_roc":     ev["auc_roc"],
                    "auc_pr":      ev["auc_pr"]},
                   f"outputs/{safe_name}.pt")
    print("  Checkpoints saved.")


if __name__ == "__main__":
    main()

Device: cuda
Loading dataset ...
  Train (SM only): 162928 jets
  Test  (SM+BSM) : 60253 jets  (40732 SM + 19521 BSM)
  Input feature dim: 3

  Training: ChebNet-GLAD
